# DataQualityAgent — «Детектив данных»

Ноутбук демонстрирует работу `DataQualityAgent` — агента, который автоматически
выявляет и устраняет проблемы качества данных.

**Три части:**
1. **Детектив** — обнаружение пропусков, дубликатов, выбросов, дисбаланса классов + визуализация.
2. **Хирург** — сравнение нескольких стратегий чистки.
3. **Аргумент** — обоснование выбора лучшей стратегии для ML-задачи.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from agents.data_quality_agent import DataQualityAgent

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

%matplotlib inline

## 0. Загрузка данных и внесение реалистичных проблем

Берём `sentiment_dataset.csv`, собранный DataCollectionAgent (Задание 1).
Чтобы продемонстрировать возможности агента, внесём контролируемые проблемы качества —
пропуски, дубликаты и выбросы по длине текста.

In [ ]:
DATA_PATH = '../../DataCollectionAgent/data/raw/sentiment_dataset.csv'

df_raw = pd.read_csv(DATA_PATH)
print(f'Загружено: {df_raw.shape}')
df_raw.head()

In [ ]:
rng = np.random.RandomState(42)
df = df_raw.copy()

# 1) Пропуски — ~5 % в text, ~3 % в label
miss_text = rng.choice(df.index, size=int(len(df) * 0.05), replace=False)
miss_label = rng.choice(df.index, size=int(len(df) * 0.03), replace=False)
df.loc[miss_text, 'text'] = np.nan
df.loc[miss_label, 'label'] = np.nan

# 2) Дубликаты — ~4 % строк
dup_idx = rng.choice(df.dropna().index, size=int(len(df) * 0.04), replace=False)
df = pd.concat([df, df.loc[dup_idx]], ignore_index=True)

# 3) Искусственный дисбаланс — удалим часть positive, чтобы ratio < 0.5
pos_idx = df[df['label'] == 'positive'].index
drop_pos = rng.choice(pos_idx, size=int(len(pos_idx) * 0.45), replace=False)
df = df.drop(drop_pos).reset_index(drop=True)

# 4) Выбросы по длине текста — несколько очень длинных строк
for i in rng.choice(df.dropna(subset=['text']).index, size=10, replace=False):
    df.loc[i, 'text'] = df.loc[i, 'text'] * 15  # повтор текста 15 раз

print(f'Датасет с внесёнными проблемами: {df.shape}')
df.info()

---
## Часть 1: Детектив — обнаружение проблем качества

Запускаем `detect_issues()` и визуализируем каждый тип проблемы.

In [ ]:
agent = DataQualityAgent(outlier_method='iqr')
report = agent.detect_issues(df)

print('=== Quality Report ===')
for key in ('missing', 'duplicates', 'outliers', 'imbalance'):
    print(f'\n{key}:')
    val = report[key]
    if isinstance(val, list):
        for item in val:
            print(f'  {item}')
    elif isinstance(val, dict):
        for k, v in val.items():
            print(f'  {k}: {v}')
    else:
        print(f'  {val}')

### 1.1 Пропущенные значения

In [ ]:
missing_data = report['missing']

if missing_data:
    cols = list(missing_data.keys())
    counts = [missing_data[c]['count'] for c in cols]
    pcts = [missing_data[c]['percent'] for c in cols]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].barh(cols, counts, color=sns.color_palette('Reds_d', len(cols)))
    axes[0].set_xlabel('Количество пропусков')
    axes[0].set_title('Пропуски — абсолютные значения')
    for i, v in enumerate(counts):
        axes[0].text(v + 1, i, str(v), va='center')

    axes[1].barh(cols, pcts, color=sns.color_palette('Oranges_d', len(cols)))
    axes[1].set_xlabel('% от общего числа строк')
    axes[1].set_title('Пропуски — процент')
    for i, v in enumerate(pcts):
        axes[1].text(v + 0.1, i, f'{v:.1f}%', va='center')

    plt.tight_layout()
    plt.show()
else:
    print('Пропусков не обнаружено.')

### 1.2 Дубликаты

In [ ]:
dup_info = report['duplicates']

fig, ax = plt.subplots(figsize=(5, 4))
labels = ['Уникальные', 'Дубликаты']
sizes = [len(df) - dup_info['count'], dup_info['count']]
colors = ['#4CAF50', '#F44336']
ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%',
       startangle=90, textprops={'fontsize': 12})
ax.set_title(f'Дубликаты: {dup_info["count"]} из {len(df)} ({dup_info["percent"]}%)')
plt.tight_layout()
plt.show()

### 1.3 Выбросы (IQR по длине текста)

In [ ]:
if report['outliers']:
    for out in report['outliers']:
        col = out['column']
        if col.endswith('_length'):
            src_col = col.replace('_length', '')
            series = df[src_col].dropna().str.len()
            label = f'Длина текста (столбец «{src_col}»)'
        else:
            series = df[col].dropna()
            label = col

        fig, axes = plt.subplots(1, 2, figsize=(13, 4))

        axes[0].boxplot(series, vert=False)
        axes[0].axvline(out['lower_bound'], color='red', ls='--', label=f'Lower={out["lower_bound"]:.0f}')
        axes[0].axvline(out['upper_bound'], color='red', ls='--', label=f'Upper={out["upper_bound"]:.0f}')
        axes[0].set_title(f'Boxplot — {label}')
        axes[0].legend(fontsize=9)

        axes[1].hist(series, bins=50, edgecolor='white', alpha=0.8)
        axes[1].axvline(out['lower_bound'], color='red', ls='--')
        axes[1].axvline(out['upper_bound'], color='red', ls='--')
        axes[1].set_title(f'Распределение — {label}')
        axes[1].set_xlabel(label)
        axes[1].set_ylabel('Частота')

        plt.suptitle(
            f'Выбросов: {out["count"]} ({out["percent"]}%) | '
            f'Метод: {out["method"].upper()}, bounds=[{out["lower_bound"]:.0f}, {out["upper_bound"]:.0f}]',
            fontsize=11, y=1.02
        )
        plt.tight_layout()
        plt.show()
else:
    print('Выбросов не обнаружено.')

### 1.4 Дисбаланс классов

In [ ]:
imb = report['imbalance']

if imb:
    dist = imb['distribution']
    classes = list(dist.keys())
    counts = list(dist.values())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    palette = sns.color_palette('Set2', len(classes))
    axes[0].bar(classes, counts, color=palette)
    axes[0].set_ylabel('Количество')
    axes[0].set_title(f'Распределение классов (столбец «{imb["column"]}»)')
    for i, v in enumerate(counts):
        axes[0].text(i, v + 5, str(v), ha='center', fontsize=10)

    axes[1].pie(counts, labels=classes, autopct='%1.1f%%', colors=palette, startangle=90)
    axes[1].set_title(
        f'Imbalance ratio: {imb["imbalance_ratio"]:.3f} '
        f'({"⚠️ дисбаланс" if imb["is_imbalanced"] else "✓ баланс"})'
    )

    plt.tight_layout()
    plt.show()
else:
    print('Информация о дисбалансе отсутствует.')

---
## Часть 2: Хирург — стратегии чистки и сравнение

Применим **две** стратегии чистки и сравним результаты.

### Стратегия A: Консервативная (drop + clip_iqr)

Удаляем строки с пропусками и дубликаты, обрезаем выбросы (clip по IQR).
Сохраняет максимум данных — выбросы не удаляются, а приводятся к границам.

In [ ]:
strategy_a = {
    'missing': 'drop',
    'duplicates': 'drop',
    'outliers': 'clip_iqr',
}

df_clean_a = agent.fix(df, strategy=strategy_a)
print(f'Стратегия A: {df.shape} → {df_clean_a.shape}')

### Стратегия B: Агрессивная (mode + drop_iqr + oversample)

Заполняем пропуски модой, удаляем выбросы целиком, компенсируем дисбаланс oversampling'ом.

In [ ]:
strategy_b = {
    'missing': 'mode',
    'duplicates': 'drop',
    'outliers': 'drop_iqr',
    'imbalance': 'oversample',
}

df_clean_b = agent.fix(df, strategy=strategy_b)
print(f'Стратегия B: {df.shape} → {df_clean_b.shape}')

### Сравнительная таблица: Стратегия A vs Стратегия B

In [ ]:
comp_a = agent.compare(df, df_clean_a)
comp_b = agent.compare(df, df_clean_b)

comparison = comp_a[['metric', 'before']].copy()
comparison.columns = ['Метрика', 'До чистки']
comparison['Стратегия A (после)'] = comp_a['after']
comparison['Δ A'] = comp_a['delta']
comparison['Стратегия B (после)'] = comp_b['after']
comparison['Δ B'] = comp_b['delta']

print('\n══════════════════════════════════════════════════════════════')
print('  Сравнение стратегий: A (консервативная) vs B (агрессивная)')
print('══════════════════════════════════════════════════════════════\n')
comparison

In [ ]:
key_metrics = comparison[
    comparison['Метрика'].isin(['rows', 'missing_values_total', 'duplicates', 'outliers_total', 'imbalance_ratio'])
].copy()

fig, axes = plt.subplots(1, len(key_metrics), figsize=(4 * len(key_metrics), 5))
if len(key_metrics) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, key_metrics.iterrows()):
    metric = row['Метрика']
    vals = [row['До чистки'], row['Стратегия A (после)'], row['Стратегия B (после)']]
    labels_bar = ['До', 'A', 'B']
    colors = ['#9E9E9E', '#2196F3', '#FF9800']
    bars = ax.bar(labels_bar, vals, color=colors, edgecolor='white', linewidth=1.5)
    ax.set_title(metric, fontsize=11)
    for bar, val in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f'{val}', ha='center', va='bottom', fontsize=9
        )

plt.suptitle('Ключевые метрики: До / Стратегия A / Стратегия B', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Визуализация распределения классов после чистки

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (title, data) in zip(axes, [
    ('До чистки', df),
    ('Стратегия A', df_clean_a),
    ('Стратегия B', df_clean_b),
]):
    if 'label' in data.columns:
        vc = data['label'].value_counts()
        ax.bar(vc.index.astype(str), vc.values, color=sns.color_palette('Set2'))
        ax.set_title(title)
        ax.set_ylabel('Количество')
        for i, v in enumerate(vc.values):
            ax.text(i, v + 5, str(v), ha='center', fontsize=9)

plt.suptitle('Распределение классов: до и после каждой стратегии', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Визуализация распределения длины текста после чистки

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, (title, data) in zip(axes, [
    ('До чистки', df),
    ('Стратегия A (clip)', df_clean_a),
    ('Стратегия B (drop)', df_clean_b),
]):
    lengths = data['text'].dropna().str.len()
    ax.hist(lengths, bins=50, edgecolor='white', alpha=0.8, color='steelblue')
    ax.set_title(title)
    ax.set_xlabel('Длина текста')
    ax.axvline(lengths.median(), color='red', ls='--', label=f'Медиана={lengths.median():.0f}')
    ax.legend(fontsize=9)

axes[0].set_ylabel('Частота')
plt.suptitle('Распределение длины текста: до и после чистки', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## Часть 3: Аргумент — обоснование выбора стратегии

### Задача

Наша ML-задача — **бинарная классификация тональности текста** (positive / negative).
Модель должна корректно определять эмоциональную окраску рецензий на фильмы.

### Анализ стратегий

| Критерий | Стратегия A (консервативная) | Стратегия B (агрессивная) |
|---|---|---|
| Пропуски | **drop** — удаляем строки с NaN | **mode** — заполняем модой |
| Дубликаты | **drop** — удаляем | **drop** — удаляем |
| Выбросы | **clip_iqr** — обрезаем до границ | **drop_iqr** — удаляем строки-выбросы |
| Дисбаланс | не корректируем | **oversample** — дублируем минорный класс |

### Выбор: Стратегия A (консервативная)

Для задачи классификации тональности текста **стратегия A** предпочтительна по следующим причинам:

1. **Сохранение объёма данных.** Текстовые модели (TF-IDF + Logistic Regression, BERT) выигрывают
   от большего количества обучающих примеров. Стратегия A удаляет только строки без информации
   (NaN, дубли), но сохраняет длинные тексты, обрезая их до разумной длины.

2. **Clip лучше drop для текста.** «Выбросы» по длине текста — это длинные рецензии,
   которые содержат ценную информацию о тональности. Обрезка (clip) сохраняет начало
   текста (где обычно сосредоточен основной смысл), а полное удаление (drop) теряет
   потенциально информативные примеры.

3. **Осторожность с oversampling.** Дублирование текстовых примеров минорного класса
   (стратегия B) создаёт точные копии в обучающей выборке. Это может привести к
   переобучению модели на конкретных формулировках. Для текста лучше использовать
   взвешивание классов (`class_weight='balanced'`) при обучении, что не искажает
   распределение признаков.

4. **Заполнение модой (стратегия B) рискованно для текста.** Замена пропущенного текста
   самым частым значением создаёт шумные примеры, которые могут ухудшить качество
   классификации.

### Рекомендация

Для production-пайплайна рекомендуется **стратегия A** с дополнением:
- `class_weight='balanced'` при обучении модели для компенсации дисбаланса
- Мониторинг F1-macro (а не accuracy) как основной метрики
- Периодический аудит данных с помощью `detect_issues()` после каждого пополнения датасета

---
## Сохранение очищенного датасета

In [ ]:
os.makedirs('../data', exist_ok=True)

df_clean_a.to_csv('../data/sentiment_clean.csv', index=False)
print(f'Сохранено: ../data/sentiment_clean.csv ({df_clean_a.shape})')

# Итоговый отчёт сравнения
comparison.to_csv('../data/comparison_report.csv', index=False)
print(f'Отчёт сравнения: ../data/comparison_report.csv')

In [ ]:
print('\n═══ Финальная проверка очищенного датасета ═══')
final_report = agent.detect_issues(df_clean_a)
print(f'Строк: {final_report["shape"]["rows"]}')
print(f'Пропуски: {sum(v["count"] for v in final_report["missing"].values())}')
print(f'Дубликаты: {final_report["duplicates"]["count"]}')
print(f'Выбросы: {sum(o["count"] for o in final_report["outliers"])}')
if final_report['imbalance']:
    print(f'Imbalance ratio: {final_report["imbalance"]["imbalance_ratio"]:.3f}')
print('\nГотово!')